# ReAct Agent from Scratch

`bind_tools` is convenient, but it hides what's actually happening. This notebook strips it away.

We'll build a working **ReAct** (*Reasoning + Acting*) agent using only:
- `ChatOllama` as a raw text-in / text-out transport
- a structured prompt
- regex to parse the LLM's text
- a Python loop

No `bind_tools`, no `tool_calls` field, no LangChain agent helpers. At the end, we reveal the same task solved with `bind_tools` in 10 lines — and you'll understand what that abstraction does.

In [1]:
from langchain_ollama import ChatOllama
import pandas as pd
import re

llm = ChatOllama(model="llama3.2:3b", temperature=0.3)  # lower temp → more format-compliant

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tools as plain Python functions

No `@tool` decorator — just functions and a registry dict. This is the pre-framework world.

In [2]:
def lookup_order(order_no: str) -> str:
    df = pd.read_csv("order_mock_data.csv")
    if not order_no.startswith("#"):
        order_no = "#" + order_no
    row = df[df["id"] == order_no]
    if row.empty:
        return f"No order found: {order_no}"
    r = row.iloc[0]
    return f"OrderNo={r['id']}, Customer={r['name']}, Items={r['items']}, Status={r['status']}"

def lookup_customer(name: str) -> str:
    df = pd.read_csv("customers.csv")
    row = df[df["name"].str.lower() == name.lower().strip()]
    if row.empty:
        return f"No customer found: {name}"
    r = row.iloc[0]
    return f"CustomerID={r['customer_id']}, Tier={r['tier']}, TotalSpent=${r['total_spent']}"

TOOLS = {
    "lookup_order":    {"fn": lookup_order,
                        "desc": "Look up order by ID like #ORDNO5. Returns status, items, customer."},
    "lookup_customer": {"fn": lookup_customer,
                        "desc": "Look up a customer profile by full name. Returns tier and spend."},
}

def format_tools() -> str:
    return "\n".join(f"- {n}: {t['desc']}" for n, t in TOOLS.items())

## 2. The ReAct prompt — the format IS the algorithm

We teach the model the exact shape we expect. The model will produce text in this shape; our loop parses it.

In [3]:
REACT_PROMPT = """\
You are a helpful customer-support agent. Use this EXACT format to answer:

Question: the user's question
Thought: reason about what to do next
Action: the tool to call, one of [{tool_names}]
Action Input: a single string argument to pass to the tool
Observation: the tool's result (the system fills this in — do not write it yourself)
... (Thought / Action / Action Input / Observation can repeat)
Thought: I now know the final answer.
Final Answer: the answer to the user

Available tools:
{tool_descriptions}

Question: {user_query}
Thought:"""

## 3. Parsing the LLM's text output

We extract `Action:` and `Action Input:` with regex, and detect `Final Answer:` to know when we're done.

In [4]:
def parse_action(text: str):
    m = re.search(r"Action:\s*(\S+)", text)
    return m.group(1).strip() if m else None

def parse_action_input(text: str):
    m = re.search(r"Action Input:\s*(.+)", text)
    if not m:
        return None
    # take just the first line, strip quotes
    val = m.group(1).splitlines()[0].strip()
    return val.strip('"').strip("'")

def parse_final_answer(text: str):
    m = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    return m.group(1).strip() if m else None

## 4. The loop — THINK → ACT → OBSERVE → repeat

We use the `stop=["Observation:"]` trick so the LLM yields control back to us after it emits an action — otherwise it would try to hallucinate the observation itself.

In [5]:
def react_agent(user_query: str, max_iters: int = 6, verbose: bool = True) -> str:
    prompt = REACT_PROMPT.format(
        tool_names=", ".join(TOOLS.keys()),
        tool_descriptions=format_tools(),
        user_query=user_query,
    )
    history = prompt
    llm_stop = llm.bind(stop=["Observation:"])

    for i in range(max_iters):
        output = llm_stop.invoke(history).content
        history += output
        if verbose:
            print(f"\n--- Iteration {i+1} ---\n{output}")

        final = parse_final_answer(output)
        if final:
            return final

        action = parse_action(output)
        action_input = parse_action_input(output)

        if action is None or action not in TOOLS:
            obs = f"(format error — no valid Action found. Valid tools: {list(TOOLS.keys())})"
        elif action_input is None:
            obs = "(format error — Action Input missing)"
        else:
            try:
                obs = TOOLS[action]["fn"](action_input)
            except Exception as e:
                obs = f"Tool error: {e}"

        if verbose:
            print(f"Observation: {obs}")
        history += f"\nObservation: {obs}\nThought:"

    return "(max iterations reached)"

In [6]:
answer = react_agent("What's the status of Charles Martinez's order #ORDNO5?")
print("\n=== Final Answer ===")
print(answer)


--- Iteration 1 ---
Thought: I need to look up the order by ID to determine the status of Charles Martinez's order #ORDNO5.

Action: lookup_order
Action Input: #ORDNO5


Observation: OrderNo=#ORDNO5, Customer=Charles Martinez, Items=Fitness Tracker, Desk Lamp, Status=Shipping

--- Iteration 2 ---
Question: What's the status of Charles Martinez's order #ORDNO5?
Thought: I need to look up the order by ID to determine the status of Charles Martinez's order #ORDNO5.

Action: lookup_order
Action Input: #ORDNO5



Observation: OrderNo=#ORDNO5, Customer=Charles Martinez, Items=Fitness Tracker, Desk Lamp, Status=Shipping

--- Iteration 3 ---
Question: What's the status of Charles Martinez's order #ORDNO5?
Thought: I need to look up the order by ID to determine the status of Charles Martinez's order #ORDNO5.

Action: lookup_order
Action Input: #ORDNO5


Observation: OrderNo=#ORDNO5, Customer=Charles Martinez, Items=Fitness Tracker, Desk Lamp, Status=Shipping

--- Iteration 4 ---
Question: What

## 5. Small-model reality check

llama3.2:3b sometimes breaks the exact format — writes `Action: "lookup_order"` (with quotes), skips the `Action Input:` line, or tries to fake an `Observation:`. Our loop handles this by:

1. `stop=["Observation:"]` — prevents the model from hallucinating observations
2. Returning a clear error as the observation when parsing fails
3. Letting the model self-correct on the next iteration

Below: deliberately trigger a format failure and watch the recovery.

In [7]:
# Chained query — the model has to reason across 2 tools.
# With llama3.2:3b, it may fumble the first iteration. Recovery path activates.
answer = react_agent(
    "Look up order #ORDNO9, then tell me what tier the customer on that order is.",
    max_iters=8,
)
print("\n=== Final Answer ===")
print(answer)


--- Iteration 1 ---
Thought: I need to look up the order by ID and then retrieve the customer's tier and spend.

Action: lookup_order
Action Input: #ORDNO9


Observation: OrderNo=#ORDNO9, Customer=Richard Lee, Items=Gaming Pad, USB-C Hub, Laptop Stand, Ergonomic Wrist Rest, HDMI Cable 6ft, Status=Warehouse

--- Iteration 2 ---
Question: What is the tier and spend of customer Richard Lee on order #ORDNO9?

Thought: I need to look up the customer's tier and spend.

Action: lookup_customer
Action Input: Richard Lee


Observation: CustomerID=CUST009, Tier=Platinum, TotalSpent=$11200.0

--- Iteration 3 ---
Question: What is the tier and spend of customer Richard Lee on order #ORDNO9?

Thought: I now know the final answer.

Final Answer: The tier for customer Richard Lee on order #ORDNO9 is Platinum, and their total spend is $11200.0.

=== Final Answer ===
The tier for customer Richard Lee on order #ORDNO9 is Platinum, and their total spend is $11200.0.


## 6. Reveal — the same task with `bind_tools`

All of the above — the prompt, the parsing, the loop — is what `bind_tools` + `tool_calls` does for you under the hood. Here's the exact same agent in 10 lines.

In [8]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def lookup_order_t(orderNo: str) -> str:
    """Look up order by ID like #ORDNO5."""
    return lookup_order(orderNo)

@tool
def lookup_customer_t(name: str) -> str:
    """Look up a customer profile by full name."""
    return lookup_customer(name)

tools = [lookup_order_t, lookup_customer_t]
tmap = {t.name: t for t in tools}
llm_bound = llm.bind_tools(tools)

def bind_tools_agent(query: str):
    msgs = [HumanMessage(content=query)]
    for _ in range(6):
        resp = llm_bound.invoke(msgs); msgs.append(resp)
        if not resp.tool_calls:
            return resp.content
        for c in resp.tool_calls:
            msgs.append(ToolMessage(content=str(tmap[c["name"]].invoke(c["args"])), tool_call_id=c["id"]))
    return "(max iters)"

print(bind_tools_agent("What's the status of Charles Martinez's order #ORDNO5?"))

Order #ORDNO5 is currently in the shipping stage. The order includes a fitness tracker and a desk lamp, and Charles Martinez can expect to receive his package within the next 3-5 business days.


## Summary

| | ReAct from scratch | `bind_tools` equivalent |
|---|---|---|
| Format contract | Text template + regex parsing | JSON schema + typed `tool_calls` |
| Failure mode | Model breaks format | Malformed JSON (rare on trained models) |
| Portability | Works on any LLM that emits text | Needs tool-use-trained model |
| Code | ~40 lines | ~10 lines |

**You now understand what LangChain's `create_react_agent` does under the hood.** The abstraction is useful precisely because you can reason about what it's replacing.